In [ ]:
import chromadb , json, ollama

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_collection("lufthansa")     # reopen the existing store

print("Loaded collection with", collection.count(), "docs")

Loaded collection with 185 docs


In [ ]:
PATH = "lufthansa_labeled.json"
labeled = json.load(open(PATH, encoding="utf-8"))

for d in labeled:
    if d["category"] == "opportunity":
        r = ollama.chat(model="llama3.1:8b", messages=[{"role": "user", "content":
            "Rate the business impact of this opportunity for Lufthansa. "
            "Answer with exactly ONE word — High, Medium, or Low.\n\n" + d["text"]}])
        ans = r["message"]["content"].strip().split()[0].strip(".,").capitalize()
        d["impact"] = ans if ans in ("High", "Medium", "Low") else "Medium"
        print(d["text"][:55], "->", d["impact"])

json.dump(labeled, open(PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Saved impact for", sum(d["category"] == "opportunity" for d in labeled), "opportunities")

Where Lufthansa’s quadjets are flying this summer. As L -> Medium
Success Booking Lufthansa via Lifemiles, here are few t -> Medium
Barclays Lufthansa Miles & More Increased Sign Up Bonus -> Medium
Financial results | AIR FRANCE KLM. 1st Quarter results -> Medium
KLM Royal Dutch Airlines - Book flights online - KLM US -> Medium
Air France KLM targets high tech cargo with. - Air Carg -> Medium
Why EasyJet's Share Price Has Surged 21% in a Month — A -> Medium
Emirates boosts long-haul fleet with mega order - PaxEx -> Medium
Turkish Airlines - airBaltic Expansion. Turkish Airline -> Medium
Award-winning Lufthansa Allegris cabin now bookable for -> Medium
Media Library - newsroom.lufthansagroup.com. Lufthansa  -> Medium
Investor Relations - Lufthansa Group Investor Relations -> Medium
Saved impact for 12 opportunities


In [19]:
def ceo_agent(question, k=5):
    # 1. RETRIEVE evidence
    results   = collection.query(query_texts=[question], n_results=k)
    retrieved = results["documents"][0]
    metas     = results["metadatas"][0]
    context   = "\n\n".join(f"[{m['source']}] {doc}" for doc, m in zip(retrieved, metas))

    # 2. PROMPT
    system_prompt = """You are a strategic advisor to the CEO of Lufthansa.
Use ONLY the evidence provided — do not invent facts.
Return a JSON object with EXACTLY these keys:
- "recommendation": one clear strategic action (string)
- "justification": 1-2 sentences explaining WHY this recommendation follows from the evidence
- "supporting_evidence": list of 2-3 short evidence points from the context
- "expected_impact": expected business impact (string)
- "risk_level": one of "High", "Medium", "Low"
- "priority": one of "High", "Medium", "Low"
"""
    user_prompt = f"Evidence:\n{context}\n\nQuestion: {question}"

    # 3. GENERATE (structured JSON)
    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        format="json"
    )

    # 4. PARSE + attach question and evidence URLs
    rec = json.loads(response["message"]["content"])
    rec["question"] = question
    rec["sources"]  = [m["url"] for m in metas]
    return rec

In [20]:
questions = [
    "What are the major opportunities for Lufthansa?",
    "What are the biggest risks for Lufthansa?",
    "What are competitors doing?",
    "Which technologies or trends should Lufthansa management monitor?",
    "What strategic actions should Lufthansa prioritize?",
]

recommendations = []
for q in questions:
    print("Generating:", q)
    recommendations.append(ceo_agent(q))

json.dump(recommendations,
          open("recommendations.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("\n Saved", len(recommendations), "recommendations")

Generating: What are the major opportunities for Lufthansa?
Generating: What are the biggest risks for Lufthansa?
Generating: What are competitors doing?
Generating: Which technologies or trends should Lufthansa management monitor?
Generating: What strategic actions should Lufthansa prioritize?

 Saved 5 recommendations


CEO Briefing (Section 7)

In [21]:
def ceo_briefing(recommendations):
    # summarize the recommendations as input
    rec_summary = "\n".join(
        f"- {r['recommendation']} (priority {r['priority']}, risk {r['risk_level']})"
        for r in recommendations
    )

    system_prompt = """You are chief of staff to the CEO of Lufthansa.
Write a concise executive briefing as a JSON object with EXACTLY these 3 keys:
- "what_happened": a single plain-text string (2-3 sentences)
- "why_it_matters": a single plain-text string (2-3 sentences)
- "what_to_do_next": a single plain-text string (2-3 sentences)
Each value MUST be a plain string — NOT a nested object, dict, or list.
Base it ONLY on the recommendations provided. Do not invent facts."""

    user_prompt = f"Strategic recommendations:\n{rec_summary}\n\nWrite the CEO briefing."

    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        format="json"
    )
    return json.loads(response["message"]["content"])

In [ ]:
briefing = ceo_briefing(recommendations)
json.dump(briefing, open("ceo_briefing.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print(json.dumps(briefing, indent=2, ensure_ascii=False))

{
  "what_happened": "Recent customer surveys have revealed significant dissatisfaction with our service, impacting loyalty and revenue.",
  "why_it_matters": "These issues are eroding our market share and damaging our reputation, threatening long-term sustainability.",
  "what_to_do_next": "We will immediately implement the recommended strategies to improve customer experience, address satisfaction concerns, monitor the competitive landscape, integrate Lufthansa Cargo with global logistics networks, and invest in data collection from fuel-efficient aircraft."
}


### REWORK

In [23]:
from retrieval import semantic_search, bm25_search, hybrid_search

In [ ]:
def retrieve_evidence(query, k_each=5, final_k=5):
    """Run all 3 retrievers, pool results, dedup by URL, keep the best by consensus."""
    methods = {
        "semantic": semantic_search(query, k_each),
        "bm25":     bm25_search(query, k_each),
        "hybrid":   hybrid_search(query, k_each),
    }

    seen = {}                                   # url -> {doc, hits, rank_sum}
    for docs_list in methods.values():
        for rank, d in enumerate(docs_list):    # rank 0 = top of that method
            key = d["url"]
            if key not in seen:
                seen[key] = {"doc": d, "hits": 0, "rank_sum": 0}
            seen[key]["hits"]     += 1           # how many methods found it
            seen[key]["rank_sum"] += rank        # how high they ranked it

    # best = found by MOST methods, tie-break by best average rank
    ranked = sorted(seen.values(), key=lambda x: (-x["hits"], x["rank_sum"]))
    return [x["doc"] for x in ranked[:final_k]]

In [25]:
import ollama, json

def make_plan(goal):
    """Break the CEO's abstract goal into specific, searchable sub-questions."""
    system = (
        "You are a research planner for a strategic intelligence agent about Lufthansa. "
        "Break the user's question into 2-4 SPECIFIC, keyword-rich sub-questions "
        "that will retrieve good evidence from a news database. "
        'Return JSON exactly like: {"steps": ["...", "...", "..."]}'
    )
    res = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": goal},
        ],
        format="json",
    )
    return json.loads(res["message"]["content"])["steps"]

In [26]:
goal = "What are the biggest risks for Lufthansa?"
plan = make_plan(goal)

print("🧭 PLAN for:", goal)          # ← showing the agent's thinking (not a black box!)
for i, s in enumerate(plan, 1):
    print(f"   {i}. {s}")

🧭 PLAN for: What are the biggest risks for Lufthansa?
   1. What are the top 5 financial challenges facing Lufthansa in the current market?
   2. What are the most significant safety concerns related to Lufthansa's recent flight operations or fleet maintenance?
   3. How is Lufthansa addressing the increasing competition from low-cost carriers in Europe and the impact on its market share?
   4. What are the regulatory risks faced by Lufthansa due to changes in EU aviation laws or sanctions related to other countries?


In [37]:
def gather_evidence(goal, final_per_step=4):
    """Plan the goal, retrieve for each sub-question, pool + dedup the evidence."""
    plan = make_plan(goal)                              # 1. break goal into sub-questions
    print(f"🧭 PLANNING:{goal}")
    for s in plan:
        print("   -", s)

    pooled = []
    for sub_q in plan:                                  # 2. retrieve for EACH sub-question
        docs = retrieve_evidence(sub_q, final_k=final_per_step)
        print(f"   🔎 {sub_q[:45]}... → {len(docs)} docs")
        pooled.extend(docs)                             # add them all to one big list

    unique = list({d["url"]: d for d in pooled}.values())  # 3. dedup by URL
    print(f"\n📊 Pooled {len(pooled)} → {len(unique)} unique evidence docs")
    return unique

In [39]:
evidence = gather_evidence("What are the biggest risks for Lufthansa?")
print("\nSample:", evidence[0]["text"][:80])

🧭 PLANNING:What are the biggest risks for Lufthansa?
   - List recent news articles about Lufthansa's financial performance and identify any potential threats to its profitability
   - Research industry reports on the airline's market share, competition, and pricing strategies to determine vulnerabilities in its business model
   - Analyze news coverage of regulatory issues affecting airlines, such as EU emissions regulations or air traffic control disruptions, to assess risks related to compliance and operations
   - Examine articles about Lufthansa's passenger growth, route expansions, and service offerings to identify potential challenges in managing expansion and maintaining customer satisfaction
   🔎 List recent news articles about Lufthansa's f... → 4 docs
   🔎 Research industry reports on the airline's ma... → 4 docs
   🔎 Analyze news coverage of regulatory issues af... → 4 docs
   🔎 Examine articles about Lufthansa's passenger ... → 4 docs

📊 Pooled 16 → 14 unique evidence docs

In [ ]:
from retrieval import docs as labeled_docs   # the full labeled docs (have category/sentiment/severity)
from collections import Counter

label_by_url = {d["url"]: d for d in labeled_docs}   # URL → full labeled doc (lookup table)
#print (label_by_url)
def analyze(evidence):
    """Attach each doc's Task-4 label (by URL) and summarize what we found."""
    for d in evidence:
        full = label_by_url.get(d["url"], {})         # find the labeled version by URL
        d["category"]  = full.get("category", "unknown") # ('key',default if not found)
        d["sentiment"] = full.get("sentiment", "unknown")
        d["severity"]  = full.get("severity")         # only risks have this

    counts = Counter(d["category"] for d in evidence)  # how many of each category
    print("📊 ANALYSIS:")
    for cat, n in counts.items():
        print(f"   {cat}: {n}")
    return evidence, counts

{'https://investor-relations.lufthansagroup.com/en/financial-reports-publications/financial-reports.html': {'text': "Financial reports - Lufthansa Group Investor Relations. Financial reports 1st Interim Report 2026 Lufthansa Group publishes its 1st interim report 2026. Carsten Spohr, Chairman of the Executive Board and Till Streichert, Member of the Executive Board and CFO present the three month figures of 2026 in an Analysts' Conference.", 'url': 'https://investor-relations.lufthansagroup.com/en/financial-reports-publications/financial-reports.html', 'source': 'company', 'sentiment': 'neutral', 'category': 'trend', 'category_score': 0.394}, 'https://investor-relations.lufthansagroup.com/en/financial-reports-publications.html': {'text': 'Financial reports & publications - Lufthansa Group Investor Relations. The Lufthansa Group reports in detail about the sustainable commitment regarding its material topics in the non-financial declaration which is a part of the annual report with refe

In [32]:
analyzed, counts = analyze(evidence)
print("\nExample:", analyzed[0]["category"], "→", analyzed[0]["text"][:55])

📊 ANALYSIS:
   trend: 9
   risk: 6

Example: trend → Lufthansa Group Posts Record Revenue, Profit Surge. Mar


In [41]:
def decide_enough(goal, evidence, counts):
    """Ask the LLM whether the evidence found is enough to answer the goal."""
    summary = ", ".join(f"{n} {cat}" for cat, n in counts.items())   # "9 trend, 6 risk"
    sample  = " | ".join(d["text"][:70] for d in evidence[:4])       # peek at a few docs

    system = (
        "You are the decision step of a research agent about Lufthansa. "
        "Given the question and a summary of the evidence found, decide if it is ENOUGH "
        "to answer the question well. "
        'Return JSON: {"sufficient": true or false, "reason": "one short sentence"}'
    )
    res = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": f"Question: {goal}\nEvidence: {summary}\nSamples: {sample}"},
        ],
        format="json",
    )
    return json.loads(res["message"]["content"])

In [42]:
verdict = decide_enough(goal, analyzed, counts)
print("🤔 DECIDE:", verdict)

🤔 DECIDE: {'sufficient': False, 'reason': "The evidence provided primarily discusses the company's financial performance and does not specifically address the biggest risks for Lufthansa."}


In [43]:
def reformulate(goal, reason):
    """Rewrite the question to be more specific when the evidence was insufficient."""
    system = (
        "The previous search for this question did not find specific enough evidence. "
        "Rewrite it into ONE more specific, focused question that targets the missing information. "
        'Return JSON: {"new_goal": "..."}'
    )
    res = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": f"Question: {goal}\nWhy it failed: {reason}"},
        ],
        format="json",
    )
    return json.loads(res["message"]["content"])["new_goal"]

In [44]:
# 1. the agent rewrites its own question using the failure reason
new_goal = reformulate(goal, verdict["reason"])
print("🔄 REFORMULATED →", new_goal)

# 2. try again with the sharper question
evidence2 = gather_evidence(new_goal)
analyzed2, counts2 = analyze(evidence2)
verdict2 = decide_enough(new_goal, analyzed2, counts2)
print("\n🤔 DECIDE again:", verdict2)

🔄 REFORMULATED → What are the most significant operational, regulatory, or strategic challenges that contributed to Lufthansa's recent struggles?
🧭 PLANNING:What are the most significant operational, regulatory, or strategic challenges that contributed to Lufthansa's recent struggles?
   - Lufthansa's response to the COVID-19 pandemic: What were the key operational challenges faced by the airline in terms of reduced demand, travel restrictions, and crew availability?
   - Regulatory impact on European airlines: How did changes in EU aviation regulations, such as the Air Passenger Rights Regulation, affect Lufthansa's operations and profitability?
   - Lufthansa's strategic restructuring efforts: What were the key goals and strategies implemented by the airline to improve its competitiveness, reduce costs, and adapt to changing market conditions?
   - Lufthansa's exposure to global economic trends: How did macroeconomic factors such as recession, inflation, or changes in global trade po

In [45]:
def recommend(goal, evidence):
    """Generate the structured recommendation from the analyzed evidence."""
    context = "\n\n".join(
        f"[{d['source']} · {d.get('category','?')}] {d['text']}" for d in evidence
    )

    system_prompt = """You are a strategic advisor to the CEO of Lufthansa.
Use ONLY the evidence provided — do not invent facts.
Return a JSON object with EXACTLY these keys:
- "recommendation": one clear strategic action (string)
- "justification": 1-2 sentences explaining WHY this recommendation follows from the evidence
- "supporting_evidence": list of 2-3 short evidence points from the context
- "expected_impact": expected business impact (string)
- "risk_level": one of "High", "Medium", "Low"
- "priority": one of "High", "Medium", "Low"
"""
    res = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": f"Evidence:\n{context}\n\nQuestion: {goal}"},
        ],
        format="json",
    )
    rec = json.loads(res["message"]["content"])
    rec["question"] = goal
    rec["sources"]  = [d["url"] for d in evidence]
    return rec

In [46]:
rec = recommend(goal, analyzed)
print(json.dumps(rec, indent=2, ensure_ascii=False))

{
  "recommendation": "Investigate and address the underlying issues leading to frequent flight cancellations and disruptions",
  "justification": "The recent Lufthansa strike, major flight cancellations, and passenger delays suggest operational challenges that could impact customer satisfaction, loyalty, and ultimately, revenue.",
  "supporting_evidence": [
    "[news · risk] Lufthansa Strike: Major Flight Cancellations and Passenger",
    "[reddit · risk] Need Advice: Flight Delay Nightmare with Lufthansa - Reddit",
    "[reddit · risk] Lufthansa have delayed or cancelled every flight on both legs"
  ],
  "expected_impact": "Potential loss of customer trust, revenue, and market share due to operational issues.",
  "risk_level": "High",
  "priority": "Medium",
  "question": "What are the biggest risks for Lufthansa?",
  "sources": [
    "https://www.airwaysmag.com/new-post/lufthansa-group-record-revenue-profit",
    "https://travelradar.aero/lufthansa-group-releases-its-third-quarter-

demo Orchestrator

In [48]:
all_evidence = gather_evidence(goal)              # round 1
tries = 0
while not verdict["sufficient"] and tries < 2:
    goal = reformulate(goal, verdict["reason"])
    new_docs = gather_evidence(goal)              # round 2, 3...
    all_evidence += new_docs                       # ← ADD, don't replace
    all_evidence = list({d["url"]: d for d in all_evidence}.values())  # dedup
    analyzed, counts = analyze(all_evidence)
    verdict = decide_enough(goal, analyzed, counts)
    tries += 1
# recommend on all_evidence — which keeps the BEST docs from every round

🧭 PLANNING:What were the primary operational, strategic, and market-related risks that led to Lufthansa's significant challenges in 2020, as reported by reputable industry publications and internal company sources?
   - Lufthansa's COVID-19 pandemic response: What were the key operational risks identified by industry analysts?
   - How did the airline's strategic decisions contribute to its financial struggles in 2020, according to internal reports and external reviews?
   - What market trends and shifts, such as shifts in demand or changes in air travel regulations, posed significant challenges to Lufthansa's operations and revenue streams?
   🔎 Lufthansa's COVID-19 pandemic response: What ... → 4 docs
   🔎 How did the airline's strategic decisions con... → 4 docs
   🔎 What market trends and shifts, such as shifts... → 4 docs

📊 Pooled 12 → 11 unique evidence docs


### VALIDATE 

In [50]:
def validate(rec, evidence):
    """Check the recommendation is grounded in the evidence (no invented facts)."""
    ev = " | ".join(d["text"][:120] for d in evidence)
    system = (
        "You are the validation step of a research agent about Lufthansa. "
        "Check whether the recommendation and its justification are SUPPORTED by the evidence. "
        "If any claim is not backed by the evidence, it is invalid. "
        'Return JSON: {"valid": true or false, "reason": "one short sentence"}'
    )
    res = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content":
                f"Recommendation: {rec['recommendation']}\n"
                f"Justification: {rec['justification']}\n"
                f"Evidence: {ev}"},
        ],
        format="json",
    )
    return json.loads(res["message"]["content"])

In [51]:
check = validate(rec, analyzed)
print("🛡️ VALIDATE:", check)

🛡️ VALIDATE: {'valid': False, 'reason': 'Only the Lufthansa strike and its impact are mentioned as evidence; other claims about flight cancellations and disruptions are unsubstantiated.'}


### THE ORCHESTRATOR

In [52]:
def run_agent(goal, max_tries=2):
    print(f"\n🎯 GOAL: {goal}\n")

    # round 1: gather → analyze → decide
    evidence = gather_evidence(goal)
    analyzed, counts = analyze(evidence)
    verdict = decide_enough(goal, analyzed, counts)
    print("🤔 DECIDE:", verdict["sufficient"], "—", verdict["reason"])

    # self-correction loop: capped, ACCUMULATING evidence (your idea + the drift fix)
    tries = 0
    while not verdict["sufficient"] and tries < max_tries:
        print(f"\n🔄 Not enough — reformulating (try {tries+1}/{max_tries})")
        sharper = reformulate(goal, verdict["reason"])
        evidence += gather_evidence(sharper)                          # ADD, don't replace
        evidence = list({d["url"]: d for d in evidence}.values())     # dedup
        analyzed, counts = analyze(evidence)
        verdict = decide_enough(goal, analyzed, counts)               # judge vs ORIGINAL goal
        print("🤔 DECIDE:", verdict["sufficient"], "—", verdict["reason"])
        tries += 1

    # ← loop exited because SUFFICIENT or CAP hit → recommend either way (your logic!)
    print(f"\n📝 RECOMMENDING on {len(evidence)} docs...")
    rec = recommend(goal, analyzed)

    # validate (one redo if not grounded)
    check = validate(rec, analyzed)
    print("🛡️ VALIDATE:", check)
    if not check["valid"]:
        print("   ↻ regenerating...")
        rec = recommend(goal, analyzed)

    rec["evidence_sufficient"] = verdict["sufficient"]   # honest flag for the dashboard
    return rec

In [53]:
final = run_agent("What are the biggest risks for Lufthansa?")
import json
print(json.dumps(final, indent=2, ensure_ascii=False))


🎯 GOAL: What are the biggest risks for Lufthansa?

🧭 PLANNING:What are the biggest risks for Lufthansa?
   - Recent financial performance of Lufthansa in terms of revenue, profit margins, and debt levels
   - Potential impacts of air travel restrictions or changes to EU-US aviation regulations on Lufthansa's operations
   - Lufthansa's exposure to global economic trends such as recession, trade wars, and oil price fluctuations
   - Operational risks including aircraft maintenance issues, flight cancellations, and pilot shortages
   🔎 Recent financial performance of Lufthansa in ... → 4 docs
   🔎 Potential impacts of air travel restrictions ... → 4 docs
   🔎 Lufthansa's exposure to global economic trend... → 4 docs
   🔎 Operational risks including aircraft maintena... → 4 docs

📊 Pooled 16 → 14 unique evidence docs
📊 ANALYSIS:
   trend: 10
   risk: 4
🤔 DECIDE: False — The provided evidence only mentions financial successes and achievements, but does not indicate any specific risks.

🔄 